In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
import sys
sys.path.append("../")

from featurizer.pdb_feature import make_feature
from multimolecule import RnaTokenizer, RnaFmModel

In [ ]:
CSV_PATH = "resources/your_data_path"          # 数据列表
PDB_DIR = "resources/your_pdbs"            # PDB 文件目录
SAVE_DIR = "resources/rnafm_embeddings"   # Embedding 保存目录
MODEL_NAME = "resources/llm_ckpts/rnafm"  ## llm ckpt目录
DEVICE = "cuda:0"

In [ ]:
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f"创建保存目录: {SAVE_DIR}")

# 2. 加载模型和 Tokenizer
print(f"正在加载 RNA-FM 模型 ({MODEL_NAME}) 到 {DEVICE}...")
try:
    tokenizer = RnaTokenizer.from_pretrained(MODEL_NAME)
    model = RnaFmModel.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()
except Exception as e:
    print(f"加载模型失败: {e}")

In [ ]:
# 3. 读取数据列表
if not os.path.exists(CSV_PATH):
    print(f"找不到 CSV 文件: {CSV_PATH}")
    
df = pd.read_csv(CSV_PATH)
print(f"开始处理 {len(df)} 个样本...")

success_count = 0
    
# 4. 循环处理
for _, row in tqdm(df.iterrows(), total=len(df)):
    pdb_id = row['pdb_id']
    pdb_file = os.path.join(PDB_DIR, f"{pdb_id}.pdb")
    save_path = os.path.join(SAVE_DIR, f"{pdb_id}.pt")
        
    if not os.path.exists(pdb_file):
        # print(f"警告: 找不到 PDB 文件 {pdb_file}")
        continue

    # --- A. 使用 pdb_feature 提取序列 ---
    try:
        feat = make_feature(pdb_file)
        seq = feat['seq']
        from utils.common import decode_seq_tensor
        seq = decode_seq_tensor(seq)
    except Exception as e:
        print(f"解析 {pdb_id} 失败: {e}")
        continue

    if len(seq) == 0:
        continue

    # --- B. RNA-FM 推理 ---
    try:
        # 1. Tokenize (直接使用字符串，比转索引再转回来更高效)
        # RnaTokenizer 会自动处理特殊字符，通常将 'U' 视为标准 RNA
        inputs = tokenizer(seq, return_tensors="pt")
            
        # 移至 GPU
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        # 2. 模型前向
        with torch.no_grad():
            output = model(**inputs)
                
            # 3. 提取 Embedding
            # last_hidden_state shape: [1, L+2, Hidden_Dim] (包含 CLS 和 EOS)
            full_embedding = output.last_hidden_state
            # 切除首尾的 [CLS] 和 [EOS]
            # 得到 [1, L, Hidden_Dim]
            seq_embedding = full_embedding[:, 1:-1, :]    
            # 移除 batch 维度 -> [L, Hidden_Dim]
            final_embedding = seq_embedding.squeeze(0)

        # --- C. 形状校验 (Data Alignment Check) ---
        # 这一步非常重要，确保 Embedding 长度和您的坐标长度一致
        if final_embedding.shape[0] != len(seq):
            print(f"⚠️ {pdb_id} 长度不匹配! Seq: {len(seq)}, Emb: {final_embedding.shape[0]}")
            continue

        # --- D. 保存 ---
        # 保存到 CPU 以节省显存
        torch.save(final_embedding.cpu(), save_path)
        success_count += 1

    except Exception as e:
        print(f"推理 {pdb_id} 时出错: {e}")
        continue

print(f"\n✅ 处理完成！成功提取 {success_count} 个样本。")
print(f"📂 结果保存在: {SAVE_DIR}")